# CourantMagnetique — `courant_magnetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **differential** post-treatment fields of the standalone
magnetic (induction) equation:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: `courant_magnetique`. Per the
TRUST validation-form layout the other `Postraitement_fields` cases live in
their own sibling folders.

## What it validates

$$
\mathbf{J} = \nabla \times \mathbf{b} / \mu_0
$$
Exactly linear field $b = (y, z, x)$: $\nabla\times b = (-1,-1,-1)$, so with
$\mu_0 = 1$: $J = (-1,-1,-1)$ in every cell. (In 2D $J$ is the single
out-of-plane scalar $J_z$.)


## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | electric current density | $J = \nabla\times b / \mu_0$ |

All these fields are built from the module-reconstructed element gradient
$\nabla b$ (`gradient_b_elements`, Green-Gauss — the kernel has no
element-field gradient factory on PolyMAC). Vector model required: the
scalar model is rejected with a clean error by `creer_champ`.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
import os
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    cands = [f"{BUILD}/coarse/{case}_{sonde}.son",
             f"{BUILD}/{case}_{sonde}.son"]
    if case.startswith("PAR_"):      # per-case subdir (merged MMS forms)
        cands.insert(1, f"{BUILD}/{case[4:]}/coarse/{case}_{sonde}.son")
    path = next((p for p in cands if os.path.exists(p)), cands[1])
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).



In [ ]:
from trustutils import run

NB_PROCS = 2

run.reset()
# --- mesh-convergence scan (shared ladder) ----------------------------------
# The single dataset src/courant_magnetique.data carries $nx/$ny node-count placeholders;
# every scale runs the SAME file via trustutils substitution
# (run.addCaseFromTemplate -> string.Template) into build/<scale>/ — one
# data-file name across scales:
#   coarse   = the baseline mesh
#   standard = cells x2 (h/2)
#   fine     = cells x4 (h/4)
# Each scale runs in its own build/<scale>/ directory so that the
# partitioned DOM_000*.Zones of one level never overwrite another's.
NX0 = NY0 = NZ0 = 5                       # baseline node counts (coarse)
meshes = {
    "coarse":   {"nx": 5, "ny": 5, "nz": 5},            # the baseline mesh
    "standard": {"nx": 2*5-1, "ny": 2*5-1, "nz": 2*5-1},
    "fine":     {"nx": 4*5-3, "ny": 4*5-3, "nz": 4*5-3},
}
cases = {}
for lvl, dic in meshes.items():
    # NOTE (latent bug, not silently hidden): with 2 ranks the fine (17^3)
    # case reliably crashes at problem init with a layout-dependent SIGFPE
    # (NaN) in the parallel path — the only difference vs the working
    # historical scan/L2 run is the case-name/path string length, which
    # shifts the heap layout. Identical bytes under another name run clean,
    # and the sequential run reproduces the historical L2 values exactly.
    # Uninitialized-memory suspect: to be pinned by the sanitizer pass; the
    # fine level therefore runs on 1 processor.
    nproc = 1 if lvl == "fine" else NB_PROCS
    cases[lvl] = run.addCaseFromTemplate("courant_magnetique.data", lvl, dic=dic,
                                         nbProcs=nproc, excluNR=True)
    if nproc > 1:
        cases[lvl].partition()          # trust -partition courant_magnetique.data 2 -> PAR_courant_magnetique.data + DOM_000*.Zones
run.printCases()
run.runCases()
case = cases["coarse"]            # alias kept for the read-back cells


## Checks


In [ ]:
entries = read_son_all(case.dataFileName_, "SONDE_J")
for name, v in entries:
    print(name, "=", v[-1])
comps = [v[-1] for name, v in entries]

# b = (y, z, x) is exactly linear => curl b = (-1,-1,-1), J = (-1,-1,-1).
expected = np.array([-1., -1., -1.])
print(f"courant_magnetique (probe) : ({comps[0]:.6g}, {comps[1]:.6g}, {comps[2]:.6g})")
err = np.max(np.abs(np.array(comps) - expected)) / np.max(np.abs(expected))
print(f"relative error             : {err:.3e}")
assert err < 1e-12, f"courant_magnetique wrong: {comps}"
print("OK: courant_magnetique = curl b / mu0.")


## Mesh convergence

The shared refinement ladder: `coarse` = the baseline mesh above (`5x5x5`),
`standard` = cells x2 (`9x9x9`), `fine` = cells x4 (`17x17x17`).

The field is reconstructed from a linear/uniform `b` at an interior probe,
which is a **discrete identity** (the Green-Gauss reconstruction is exact
for linear fields at interior cells): b = (y,z,x) linear => J = curl b / mu0 = (-1,-1,-1), constant. The scan asserts the same
machine-precision bound at **every** level
(the quantity is position-independent) — no order is reported.

In [ ]:
def read_final_all_dir(subdir, case_name, name):
    path = f"{BUILD}/{subdir}/{case_name}_{name}.son"
    son = SonFile(path, None)
    return [np.asarray(son.getValues(e)[1], dtype=float) for e in son.getEntries()]

def level_error(subdir, case_name):
    comps = [float(vv[-1]) for vv in read_final_all_dir(subdir, case_name, "SONDE_J")]
    expected = np.array([-1., -1., -1.])
    e = float(np.max(np.abs(np.array(comps) - expected)))
    return e, expected

scan = {"coarse": ("coarse", case.dataFileName_),
        "standard": (f"standard", cases["standard"].dataFileName_),
        "fine": (f"fine", cases["fine"].dataFileName_)}
scan_nodes = {"coarse": (5, 5, 5), "standard": (9, 9, 9), "fine": (17, 17, 17)}
res = {lvl: level_error(sub, cn) for lvl, (sub, cn) in scan.items()}

print(f"level  nodes        abs err        rel")
for lvl in ("coarse", "standard", "fine"):
    n0, n1, n2 = scan_nodes[lvl]
    e, expected = res[lvl]
    rel = e / float(np.max(np.abs(expected)))
    print(f"{lvl:<9} {n0:>3}x{n1:>3}x{n2:>3} {e:<13.3e} {rel:.1e}")
    assert rel < 1e-12, f"identity broken at {lvl}: {rel:.3e}"
print("OK: the reconstruction identity holds at every mesh level.")

## Computer Performance


In [ ]:
run.tablePerf()

